### Instruction by instruction

## Imports

In [1]:
from typing import Any

from kirin.ir import Method
import matplotlib.pyplot as plt
import numpy as np

from bloqade import squin, tsim
from bloqade.cirq_utils import emit_circuit, load_circuit, noise
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
import matplotlib.pyplot as plt

# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)

## Team Synthesis

#### Team Synthesis Part 1: Learn the language of Clifford+ $T$

##### Team Synthesis Part 1a: Hadamard (single-qubit gate)

In [2]:
@squin.kernel
def h_state() -> Measurement:
    qubits = squin.qalloc(1)
    squin.h(qubits[0])
    bits = squin.broadcast.measure(qubits)
    return bits

In [3]:
pyqrack_target = StackMemorySimulator(min_qubits=1)
task = pyqrack_target.task(h_state)
batch_results = task.batch_run(shots=1000)
print(batch_results)
show_circuit(h_state)

{(<Measurement.Zero: 0>,): 0.511, (<Measurement.One: 1>,): 0.489}


##### Team Synthesis Part 1b: CNOT (2-qubit gate)

In [4]:
@squin.kernel
def cnot_no_op() -> Measurement:
    qubits = squin.qalloc(2)
    squin.cx(qubits[0],qubits[1])
    bits = squin.broadcast.measure(qubits)
    return bits

In [5]:
pyqrack_target = StackMemorySimulator(min_qubits=2)
task = pyqrack_target.task(cnot_no_op)
batch_results = task.batch_run(shots=1000)
print(batch_results)
show_circuit(cnot_no_op)

{(<Measurement.Zero: 0>, <Measurement.Zero: 0>): 1.0}


In [6]:
@squin.kernel
def cnot_cntrl_on() -> Measurement:
    qubits = squin.qalloc(2)
    squin.x(qubits[0])
    squin.cx(qubits[0],qubits[1])
    bits = squin.broadcast.measure(qubits)
    return bits

In [7]:
pyqrack_target = StackMemorySimulator(min_qubits=2)
task = pyqrack_target.task(cnot_cntrl_on)
batch_results = task.batch_run(shots=1000)
print(batch_results)
show_circuit(cnot_no_op)

{(<Measurement.One: 1>, <Measurement.One: 1>): 1.0}


#### Team Synthesis Part 2: Synthesize the rotation family

##### For this part, I'm thinking of looking at the actual matrices for the independent rotations and seeing if I can form a complex rotation as a result of mulitple rotations on top of eachother...?

$$
R_z(\pi/2^n), \qquad n \in \{0,1,2,3,4,5\}.
$$

##### In general, an Rz gate is described as such


$$
R_z(\theta) = 
\begin{pmatrix} 
e^{-i\theta/2} & 0 \\ 
0 & e^{i\theta/2} 
\end{pmatrix}
$$


Clifford + T gates include the following: $H, S, T,$ and $CNOT$



### Hadamard & Phase (S)
$$
H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \quad 
S = \begin{pmatrix} 1 & 0 \\ 0 & i \end{pmatrix}
$$

### CNOT (CX)
$$
CX = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}
$$

### T-Gate (Non-Clifford)
$$
T = \begin{pmatrix} 1 & 0 \\ 0 & e^{i\pi/4} \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & \frac{1+i}{\sqrt{2}} \end{pmatrix}
$$

#### We can make a kernel that does Rz and test a bunch of states?

In [8]:
@squin.kernel
def rotational_z(theta) -> Measurement:
    qubits = squin.qalloc(1)
    squin.x(qubits[0])
    squin.rz(theta, qubits[0])
    bits = squin.broadcast.measure(qubits)
    return bits

In [9]:
pyqrack_target = StackMemorySimulator(min_qubits=1)
theta = np.pi/10
task = pyqrack_target.task(rotational_z, (theta,))
batch_results = task.batch_run(shots=1000)
print(batch_results)
# show_circuit(lambda: rotational_z(theta))


{(<Measurement.One: 1>,): 1.0}


#### 180 degree rotation (n=0)

$$
R_z(\pi) = 
\begin{pmatrix} 
e^{-i\pi/2} & 0 \\ 
0 & e^{i\pi/2} 
\end{pmatrix}
$$

Using Euler's formula
$$
e^{j\theta} = \cos(\theta) + j\sin(\theta) \\
e^{i\pi/2} = \cos(\pi/2) + i\sin(\pi/2) \\
= +i
$$
Same for $e^{-i\pi/2}$ except the value is -i, we get a final matrix of 
$$
R_z(\pi) = 
\begin{pmatrix} 
1 & 0 \\ 
0 & -1 
\end{pmatrix}
$$
That is the Z-matrix. That's not in our Clifford + T, but, since I did this second (after 90 degrees) we can just apply 2 $S$ matrices and call it a day. 

#### 90 degree rotation (n=1)


$$
R_z(\pi/2) = 
\begin{pmatrix} 
e^{-i\pi/4} & 0 \\ 
0 & e^{i\pi/4} 
\end{pmatrix}
$$


Using Euler's formula
$$
e^{j\theta} = \cos(\theta) + j\sin(\theta) \\
e^{i\pi/4} = \cos(\pi/4) + i\sin(\pi/4) \\
= \sqrt2 /2 + i \sqrt2 / 2
$$

$e^{-j\theta}$ is the same just with a $-i\sin$, 

plug that into the matrix, factor out the sqrt2/2 and the (1-i), simplify


$$
R_z(\pi/2) = 
\begin{pmatrix} 
1 & 0 \\ 
0 & i 
\end{pmatrix}
$$
Which is the $S$ matrix!

#### 45 degree rotation (n=2)

Start with the definition
$$
R_z(\pi/4) = 
\begin{pmatrix} 
e^{-i\pi/8} & 0 \\ 
0 & e^{i\pi/8} 
\end{pmatrix}
$$
Factor out a $e^{-i\pi/8}$

$$
R_z(\pi/4) = 
\begin{pmatrix} 
1 & 0 \\ 
0 & e^{i\pi/8} / e^{-i\pi/8} 
\end{pmatrix}
$$
Simplify
$$
R_z(\pi/4) = 
\begin{pmatrix} 
1 & 0 \\ 
0 & e^{2i\pi/8} 
\end{pmatrix}
$$
Final push
$$
R_z(\pi/4) = 
\begin{pmatrix} 
1 & 0 \\ 
0 & e^{i\pi/4}
\end{pmatrix}
$$
And that's the $T$ matrix!

#### 22.5 degree rotation (n=3)

$$
R_z(\pi/8) = 
\begin{pmatrix} 
e^{-i\pi/16} & 0 \\ 
0 & e^{i\pi/16} 
\end{pmatrix}
$$

After factoring and all cool things, 
$$
R_z(\pi/8) = 
\begin{pmatrix} 
1 & 0 \\ 
0 & e^{i\pi/8} 
\end{pmatrix}
$$
This doesn't really resemble...anything? As in, this is not EXACTLY described by any of the matrices in Clifford + $T$, so we have to approximate!

IF we're approximating, let's consider the set that we have. Clifford + $T$ is $H,S,T$ and $CNOT$. Looking at some visualizers, our team tried to understand how to approach this 22.5 degree rotation. We noticed in visual tools that the Hadamard, S, T matrix interleaved was getting...closer to the right answer...

After the initial visualization and some matrix calculations by hand on a whiteboard, we realized that we might want to reconsider what we're doing. We wanted to make an algorithm to try and run different combinations of matrices.

Initially, we thought of doing a "gradient descent" like problem solution, where we would pick the best gate in the set that would drop the distance as much as possible, in a "greedy algorithm" manner. Then, we considered the possibiltiy that a locally-unoptimal gate might lead to better solutions later, so we decided to implement a "future looking" to the algorithm. Since the gate set is small enough, we can afford to look ahead by a step. Finally, we decided that we can make a hyperparameter titled "future" which would decide just how far you could go with your prediction.

Before we went and designed this, however, we had to consider a special constraint for this problem. From the challenge, we know that Non-Clifford gates are expensive, so we wanted to consider that when designed an algorithm that would pick the best quantum gate adaptively and try to prioritize the *other* gates instead. 

In [5]:
from itertools import product as iter_product

#using the formula given
def distance(A, B):
    A_conj_tp = A.conj().T
    val = np.abs(np.trace(A_conj_tp @ B)) / 2
    val = min(val, 1.0)
    return np.sqrt(1 - val)


In [9]:
def rz_matrix(theta):
    return np.array([
        [np.exp(-1j * theta / 2), 0],
        [0, np.exp(1j * theta / 2)]
    ], dtype=complex)

In [ ]:
import numpy as np


H = 1/np.sqrt(2) * np.array([[1, 1], [1, -1]], dtype=complex)
S = np.array([[1, 0], [0, 1j]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)
T_adj = T.conj().T
S_adj = S.conj().T

gate_set = {
    "H": H,
    "S": S,
    "T": T,
    "T†": T_adj,
    "S†": S_adj,
    "HT": H@T,
}


from collections import deque

def brute_force_synthesis(target_matrix, tolerance=1e-6, max_depth=12):
    """
    Breadth-first search (BFS) to find the shortest gate sequence natively.
    Guarantees finding the absolute shortest sequence within max_depth.
    """
    # Queue stores: (current_matrix, sequence_of_gates)
    queue = deque([(np.eye(2, dtype=complex), [])])
    
    while queue:
        current_matrix, sequence = queue.popleft()
        
        # Check if we met the strict tolerance
        if distance(target_matrix, current_matrix) < tolerance:
            return sequence, current_matrix
            
        # Stop expanding this branch if we hit the depth limit
        if len(sequence) >= max_depth:
            continue
            
        last_gate = sequence[-1] if sequence else None
        
        for gate_name, gate_mat in gate_set.items():
            # Optimization: Skip trivial inverses to save massive amounts of compute time!
            if last_gate:
                if (last_gate == 'S' and gate_name == 'S†') or \
                   (last_gate == 'S†' and gate_name == 'S') or \
                   (last_gate == 'T' and gate_name == 'T†') or \
                   (last_gate == 'T†' and gate_name == 'T') or \
                   (last_gate == 'H' and gate_name == 'H'):
                    continue
            
            next_matrix = gate_mat @ current_matrix
            new_sequence = sequence + [gate_name]
            
            queue.append((next_matrix, new_sequence))
            
    print(f"Tolerance not reached within max_depth={max_depth}. Trying increasing depth, but be careful of memory!")
    return None, None

